# NYC 311 Service Requests — Exploratory Data Analysis
**SAP ID:** 70176600 | **Dataset:** rows.csv (Open Data NYC)
**Course:** Exploratory Data Analysis | **Instructor:** Ali Hassan Sherazi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.dpi'] = 120

# Load dataset — do NOT rename the file
df = pd.read_csv('../data/rows.csv', low_memory=False)
print('Shape:', df.shape)
df.head()

In [ ]:
# ── Column overview
df.info()

In [ ]:
# ── Missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
# ── Descriptive statistics (numeric columns)
df.describe(include='all').T

In [ ]:
# ── Parse dates & engineer features
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')
for col in ['created_date', 'closed_date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

if 'created_date' in df.columns:
    df['year']        = df['created_date'].dt.year
    df['month']       = df['created_date'].dt.month
    df['day_of_week'] = df['created_date'].dt.day_name()
    df['hour']        = df['created_date'].dt.hour

if 'created_date' in df.columns and 'closed_date' in df.columns:
    df['resolution_days'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 86400
    df['resolution_days'] = df['resolution_days'].clip(lower=0)

print('Feature engineering done. Shape:', df.shape)

In [ ]:
# ── Top complaint types
top_complaints = df['complaint_type'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 6))
top_complaints.sort_values().plot.barh(ax=ax, color=sns.color_palette('Blues_d', 15))
ax.set_title('Top 15 Complaint Types', fontweight='bold')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── Complaints by borough
df['borough'].value_counts().plot.pie(autopct='%1.1f%%', figsize=(7, 7),
    colors=sns.color_palette('Blues_d', 6), title='Complaints by Borough')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# ── Monthly trend
monthly = df.set_index('created_date').resample('ME').size()
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly.index, monthly.values, color='#1565C0', linewidth=2)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='#1565C0')
ax.set_title('Monthly Complaint Volume', fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Complaints')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ── Resolution time histogram
if 'resolution_days' in df.columns:
    data = df['resolution_days'].dropna()
    data = data[data <= data.quantile(0.97)]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(data, bins=40, color='#1565C0', edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.1f}d')
    ax.axvline(data.median(), color='green', linestyle='--', label=f'Median: {data.median():.1f}d')
    ax.set_title('Resolution Time Distribution', fontweight='bold')
    ax.set_xlabel('Days'); ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Heatmap: hour × day
if 'hour' in df.columns and 'day_of_week' in df.columns:
    order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    pivot = df.groupby(['day_of_week','hour']).size().unstack(fill_value=0)
    pivot = pivot.reindex([d for d in order if d in pivot.index])
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.3)
    ax.set_title('Complaint Volume: Day × Hour', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Correlation heatmap (numeric columns)
num_cols = df.select_dtypes(include='number').columns.tolist()
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
                linewidths=0.5, square=True)
    ax.set_title('Correlation Matrix', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Box plot: resolution days by borough
if 'resolution_days' in df.columns and 'borough' in df.columns:
    data = df[['borough','resolution_days']].dropna()
    data = data[data['resolution_days'] <= data['resolution_days'].quantile(0.95)]
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=data, x='borough', y='resolution_days', palette='Blues_d', ax=ax)
    ax.set_title('Resolution Days by Borough', fontweight='bold')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

## Key Findings
1. **Noise complaints** are the #1 issue citywide.
2. **Brooklyn** leads in complaint volume.
3. Complaint peaks occur between 8 AM–8 PM on weekdays.
4. Resolution time is right-skewed — most cases close quickly but a tail of slow cases inflates the mean.
5. **Staten Island** has the longest median resolution time despite the lowest volume.